# 03 - Análisis Bivariado

Tabla 2: características categóricas y asociación bivariada con `alteracion_osea`.

In [1]:
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from scipy.stats import chi2_contingency, fisher_exact

In [2]:
processed_dir = Path("/home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed")
parquet_path = processed_dir / "BD_Clean_Osteoporosis.parquet"
pickle_path = processed_dir / "BD_Clean_Osteoporosis.pkl"

try:
    df_clean = pd.read_parquet(parquet_path)
except Exception:
    df_clean = pd.read_pickle(pickle_path)

print(df_clean.shape)

(405, 75)


In [3]:
vars_cat = [
    "edad_cat",
    "sexo",
    "e_civil",
    "l_resid_municipio",
    "escolaridad",
    "trabaja",
    "ing_mensual_cat",
    "derechohabiente_cat",
    "enfermedad_cat",
    "vive_con_cat",
    "ayuda_enf_cod",
    "grupo_externo_enf",
    "realiza_af",
    "frecuencia_af",
    "plan_alimenticio",
    "lacteos_frecuentes",
    "imc_cat",
]

column_aliases = {
    "e_civil": "estado_civil",
    "l_resid_municipio": "lugar_residencia_municipio",
    "ayuda_enf_cod": "ayuda_enf_cat",
}

def build_categorical_bivariate_table(df, variables, outcome="alteracion_osea"):
    rows = []
    outcome_values = pd.to_numeric(df[outcome].astype("string"), errors="coerce")
    total_n = len(df)

    for variable in variables:
        column = column_aliases.get(variable, variable)
        categories = df[column].astype("string").fillna("Sin dato")
        counts = pd.crosstab(categories, outcome_values)

        for outcome_value in [0, 1]:
            if outcome_value not in counts.columns:
                counts[outcome_value] = 0

        counts = counts[[0, 1]].sort_index()
        row_percent = counts.div(counts.sum(axis=1), axis=0).mul(100)
        _, chi2_p, _, expected = chi2_contingency(counts)

        if (expected < 5).any() and counts.shape == (2, 2):
            _, p_value = fisher_exact(counts.to_numpy())
        else:
            p_value = chi2_p

        for idx, category in enumerate(counts.index):
            normal_n = int(counts.loc[category, 0])
            altered_n = int(counts.loc[category, 1])
            category_n = normal_n + altered_n
            category_pct = category_n / total_n * 100

            rows.append(
                {
                    "Variable": variable if idx == 0 else "",
                    "Categoría": category,
                    "Total (n, %)": f"{category_n} ({category_pct:.1f})",
                    "Con alteración ósea (n, %)": f"{altered_n} ({row_percent.loc[category, 1]:.1f})",
                    "Sin alteración ósea (n, %)": f"{normal_n} ({row_percent.loc[category, 0]:.1f})",
                    "p-value": round(p_value, 4) if idx == 0 else "",
                }
            )

    return pd.DataFrame(rows)

df_bivariate = df_clean.copy()
df_bivariate["frecuencia_lacteos_cat"] = pd.cut(
    pd.to_numeric(df_bivariate["frecuencia_lacteos"], errors="coerce"),
    bins=[-np.inf, 1, 2, np.inf],
    labels=["Baja (0-2 días)", "Media (3-4 días)", "Alta (5-7 días)"],
)

vars_cat_manuscrito = [
    "frecuencia_lacteos_cat" if variable == "lacteos_frecuentes" else variable
    for variable in vars_cat
]

tabla_2_final = build_categorical_bivariate_table(df_clean, vars_cat)
tabla_2_manuscrito = build_categorical_bivariate_table(df_bivariate, vars_cat_manuscrito)

variable_groups = {
    "Características demográficas": ["edad_cat", "sexo", "e_civil", "l_resid_municipio"],
    "Condiciones materiales de la vida": ["escolaridad", "trabaja", "ing_mensual_cat"],
    "Sistema sanitario": ["derechohabiente_cat", "enfermedad_cat"],
    "Cohesión social": ["vive_con_cat", "ayuda_enf_cod", "grupo_externo_enf"],
    "Estilo de vida": ["realiza_af", "frecuencia_af", "plan_alimenticio", "frecuencia_lacteos_cat", "imc_cat"],
}

variable_labels = {
    "edad_cat": "Edad",
    "sexo": "Sexo",
    "e_civil": "Estado civil",
    "l_resid_municipio": "Lugar de residencia",
    "escolaridad": "Escolaridad",
    "trabaja": "Situación laboral",
    "ing_mensual_cat": "Situación económica",
    "derechohabiente_cat": "Derechohabiente",
    "enfermedad_cat": "Enfermedad",
    "vive_con_cat": "Convivencia",
    "ayuda_enf_cod": "Apoyo en enfermedad",
    "grupo_externo_enf": "Participación en grupos de ayuda mutua",
    "realiza_af": "Actividad física",
    "frecuencia_af": "Frecuencia de actividad física",
    "plan_alimenticio": "Plan alimenticio",
    "frecuencia_lacteos_cat": "Consumo de lácteos",
    "imc_cat": "Clasificación IMC",
}

category_labels = {
    "edad_cat": {"1": "50-59", "2": "60 a 69", "3": "70 y más"},
    "sexo": {"0": "Hombre", "1": "Mujer"},
    "e_civil": {"0": "Sin pareja", "1": "En pareja"},
    "l_resid_municipio": {"0": "Municipios de Jalisco", "1": "Zona Metropolitana de Guadalajara"},
    "escolaridad": {"0": "Sin estudios", "1": "Con estudios"},
    "trabaja": {"0": "Desempleado", "1": "Con trabajo"},
    "ing_mensual_cat": {"0": "Sin información", "1": "Nivel bajo", "2": "Nivel medio", "3": "Nivel alto"},
    "derechohabiente_cat": {"0": "Sin cobertura", "1": "Con cobertura"},
    "enfermedad_cat": {"0": "Sano", "1": "Síndrome metabólico", "2": "Otras enfermedades"},
    "vive_con_cat": {"0": "Vive solo", "1": "Acompañado"},
    "ayuda_enf_cod": {"0": "Autocuidado", "1": "Tiene apoyo"},
    "grupo_externo_enf": {"0": "No participa", "1": "Sí participa"},
    "realiza_af": {"0": "No realiza", "1": "Sí realiza"},
    "frecuencia_af": {"0": "No aplica", "1": "1 vez", "2": "2-3 veces", "3": "4-5 veces", "4": "6+ veces"},
    "plan_alimenticio": {"0": "Sin plan alimenticio", "1": "Con plan alimenticio"},
    "imc_cat": {"1": "Normal", "2": "Sobrepeso", "3": "Obesidad"},
}

category_orders = {
    "edad_cat": ["1", "2", "3"],
    "sexo": ["0", "1"],
    "e_civil": ["1", "0"],
    "l_resid_municipio": ["0", "1"],
    "escolaridad": ["1", "0"],
    "trabaja": ["1", "0"],
    "ing_mensual_cat": ["1", "2", "3", "0"],
    "derechohabiente_cat": ["0", "1"],
    "enfermedad_cat": ["1", "2", "0"],
    "vive_con_cat": ["1", "0"],
    "ayuda_enf_cod": ["0", "1"],
    "grupo_externo_enf": ["0", "1"],
    "realiza_af": ["0", "1"],
    "frecuencia_af": ["4", "3", "2", "1", "0"],
    "plan_alimenticio": ["1", "0"],
    "frecuencia_lacteos_cat": ["Alta (5-7 días)", "Media (3-4 días)", "Baja (0-2 días)"],
    "imc_cat": ["1", "2", "3"],
}

def format_p_value(value):
    if value == "" or pd.isna(value):
        return ""
    p_value = float(value)
    return "< 0.001" if p_value < 0.001 else f"{p_value:.4f}"

def prepare_manuscript_rows(table):
    table = table.copy()
    table["_variable_key"] = table["Variable"].replace("", pd.NA).ffill()
    return table

def sort_variable_rows(rows, variable):
    order = category_orders.get(variable)
    if order is None:
        return rows.sort_values("Categoría")
    return rows.assign(_order=rows["Categoría"].astype(str).map({value: idx for idx, value in enumerate(order)})).sort_values("_order", na_position="last")

def render_manuscript_table(table, title):
    prepared = prepare_manuscript_rows(table)
    html = [
        "<style>",
        ".manuscript-table-wrap{max-width:1120px;overflow-x:auto;background:#fff!important;color:#111!important;color-scheme:light;padding:18px 20px;border:1px solid #d7d2c3;box-shadow:0 2px 8px rgba(0,0,0,.16);font-family:'Times New Roman',Times,serif;}",
        ".manuscript-table-wrap *{box-sizing:border-box;color:#111!important;opacity:1!important;}",
        ".manuscript-title{background:#fff!important;font-size:15px;line-height:1.35;margin:0 0 12px 0;}",
        ".manuscript-title strong{font-style:italic;}",
        ".manuscript-table{border-collapse:collapse;min-width:940px;width:100%;background:#fff!important;font-size:13.5px;line-height:1.28;table-layout:fixed;}",
        ".manuscript-table th{background:#f3f0e8!important;border-top:2px solid #111;border-bottom:1.5px solid #111;padding:8px 10px;text-align:center;font-weight:700;vertical-align:middle;}",
        ".manuscript-table td{background:#fff!important;border-bottom:1px solid #222;padding:6px 10px;vertical-align:middle;}",
        ".manuscript-table .section td{background:#efe6c8!important;border-top:1.5px solid #111;border-bottom:1.5px solid #111;font-style:italic;font-weight:700;text-align:left;padding:6px 10px;}",
        ".manuscript-table .variable{background:#fbfaf6!important;font-weight:700;width:18%;}",
        ".manuscript-table .category{width:27%;}",
        ".manuscript-table .num{text-align:center;white-space:nowrap;width:16%;}",
        ".manuscript-table .pvalue{text-align:center;white-space:nowrap;width:11%;}",
        "</style>",
        "<div class='manuscript-table-wrap'>",
        f"<p class='manuscript-title'><strong>Tabla 2.</strong> {escape(title)}</p>",
        "<table class='manuscript-table'>",
        "<thead><tr><th>Variable</th><th>Categoría</th><th>Total (n, %)</th><th>Con alteración ósea<br>(n, %)</th><th>Sin alteración ósea<br>(n, %)</th><th>p-value</th></tr></thead>",
        "<tbody>",
    ]

    for section, variables in variable_groups.items():
        html.append(f"<tr class='section'><td colspan='6'>{escape(section)}</td></tr>")

        for variable in variables:
            rows = prepared.loc[prepared["_variable_key"] == variable]
            if rows.empty:
                continue

            rows = sort_variable_rows(rows, variable)
            p_value = format_p_value(rows["p-value"].replace("", pd.NA).dropna().iloc[0])
            row_span = len(rows)

            for row_idx, (_, row) in enumerate(rows.iterrows()):
                category = str(row["Categoría"])
                category = category_labels.get(variable, {}).get(category, category)
                html.append("<tr>")
                if row_idx == 0:
                    html.append(f"<td class='variable' rowspan='{row_span}'>{escape(variable_labels.get(variable, variable))}</td>")
                html.append(f"<td class='category'>{escape(category)}</td>")
                html.append(f"<td class='num'>{escape(str(row['Total (n, %)']))}</td>")
                html.append(f"<td class='num'>{escape(str(row['Con alteración ósea (n, %)']))}</td>")
                html.append(f"<td class='num'>{escape(str(row['Sin alteración ósea (n, %)']))}</td>")
                if row_idx == 0:
                    html.append(f"<td class='pvalue' rowspan='{row_span}'>{escape(p_value)}</td>")
                html.append("</tr>")

    html.extend(["</tbody>", "</table>", "</div>"])
    return HTML("".join(html))

display(render_manuscript_table(tabla_2_manuscrito, "Características categóricas de los pacientes evaluados según alteración ósea (n = 405)"))